# 20 · Upload one video and review the output

Run All once to load the upload panel below. **Upload one short clip, then click Run this video.** No video is analyzed until you click that button or explicitly run the advanced server-path workflow.

The pipeline runs tiled DMD animal proposals → YOLO-World context proposals → bounded Gemini advice → your exact-frame review → an annotated MP4. Person/vehicle boxes stay hidden until you confirm an animal in that frame and the proposed context has a supported class check.

The default preview is six seconds at one sampled frame/second. Start with a clip under 100 MB. Selected review images are sent to the configured Gemini service when enabled; the original recording stays on the notebook server. No dispatch or aircraft command is created.

## 1 · Environment

Use the installed **Python (aerial detector)** GPU kernel. The model workers use the existing isolated environments; no dependencies or checkpoints are installed automatically. Gemini uses the same configured Vertex AI access as the research pipeline.

In [ ]:
from pathlib import Path
import sys, importlib, json
from IPython.display import display, Markdown, HTML, Video

AERIAL = Path.home() / "lifejacket-dispatch/research/aerial_detector"
if not (AERIAL / "video_review_workflow.py").exists():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "research/aerial_detector"
        if (candidate / "video_review_workflow.py").exists():
            AERIAL = candidate
            break
if not (AERIAL / "video_review_workflow.py").exists():
    raise FileNotFoundError("video_review_workflow.py must be beside dmd.py in research/aerial_detector.")
if str(AERIAL) not in sys.path:
    sys.path.insert(0, str(AERIAL))
import video_review_workflow as workflow
importlib.reload(workflow)
DATA = Path.home() / "aerial-data"
def model_python(marker):
    path = DATA / marker
    return Path(path.read_text().strip()) if path.exists() else Path(sys.executable)
PYTHON_DMD = Path(sys.executable)
PYTHON_WORLD = model_python("model-comparison-v1/env-ready.txt")
PYTHON_GEMINI = model_python("gemini-handoff-v1/env-ready.txt")
print(workflow.self_check())
print("Detector environment:", PYTHON_DMD)
print("Context environment:", PYTHON_WORLD)
print("Gemini environment:", PYTHON_GEMINI)

## 2 · Upload one video

Click Upload one video, choose a local MP4/MOV/WebM or another supported video, set the short preview window, and click Run this video. The progress log, review card and video appear in this panel.

After making a review decision, click Refresh annotated video. Open the original full-resolution frame from the review panel when small targets are difficult to inspect. The approval buttons record your real decision, not an automatic AI decision.

For files over 100 MB, use JupyterLab’s normal file upload and the advanced server-path settings below.

In [ ]:
RUN_DIR = None
VIDEO = None
def _uploaded_run_ready(run, video):
    global RUN_DIR, VIDEO
    RUN_DIR, VIDEO = run, video

UPLOAD_STATE = workflow.upload_widget(
    output_root=DATA / "video-workflow-runs",
    dmd_weights=DATA / "weights/DroneMegaDetector_RFDETRMedium_2026.10.04.pth",
    world_weights=DATA / "model-comparison-v1/yolov8s-worldv2.pt",
    python_dmd=PYTHON_DMD, python_world=PYTHON_WORLD, python_gemini=PYTHON_GEMINI,
    on_ready=_uploaded_run_ready,
)

## Advanced · Use a video already on this server

You can ignore this section when using the upload panel. Alternatively, replace VIDEO = None below with a Path to an existing recording, then run the remaining cells. Existing examples:

| Case | Path under ~/aerial-data | Start / seconds |
|---|---|---|
| Aerial sea lions | aerial-patrol-v1/raft.mp4 | 0 / 6 |
| Seal + people + truck (ground-level proxy) | combined-gemini-v1/r7-release.mp4 | 16 / 4 |
| Aerial people/vehicles control | video-patrol-v1/parking-people-4k.mp4 | 0 / 3 |

The ground-level case is not validation of aerial marine-animal/context performance.

In [ ]:
# EDIT VIDEO to test your own recording.
VIDEO = None  # Or: DATA / "aerial-patrol-v1/raft.mp4"
START_SECONDS = 0
CLIP_SECONDS = 6             # None = the entire remaining source
SAMPLE_FPS = 1
MAX_SAMPLES = 120              # Raises before a large run; never silently truncates
CONFIDENCE = 0.30
REVIEW_WINDOW_SECONDS = 6
RUN_GEMINI = True              # False = detector-only; context remains unverified
MAX_GEMINI_CALLS = 6           # A hard per-run bound; unused windows remain review-needed
NATIVE_CROPS = False           # Optional padded crops; full-frame advice is the default

DMD_WEIGHTS = DATA / "weights/DroneMegaDetector_RFDETRMedium_2026.10.04.pth"
WORLD_WEIGHTS = DATA / "model-comparison-v1/yolov8s-worldv2.pt"
OUTPUT_ROOT = DATA / "video-workflow-runs"
print("Video:", VIDEO)
print("Window:", START_SECONDS, "to", "end" if CLIP_SECONDS is None else START_SECONDS + CLIP_SECONDS, "seconds")
print("Gemini enabled:", RUN_GEMINI, "| maximum calls:", MAX_GEMINI_CALLS)

## 3 · Run the pipeline

This cell waits while separate workers run DMD → YOLO-World → Gemini → the initial video export. Progress is printed below. Repeating the same run reuses its exact cached detector and review records; it does not silently repeat failed paid requests. Different source bytes or settings create a separate run and separate human decisions.

Every review window remains available even if Gemini says no, fails, or is not called because of the budget. If a worker fails, inspect the stage log printed in the error. Failed or missing review is not animal absence.

In [ ]:
if VIDEO is None:
    print("Use the upload panel above, or set VIDEO to an existing server file.")
else:
    RUN_DIR = workflow.prepare(
        VIDEO, output_root=OUTPUT_ROOT, dmd_weights=DMD_WEIGHTS, world_weights=WORLD_WEIGHTS,
        sample_fps=SAMPLE_FPS, start_seconds=START_SECONDS, clip_seconds=CLIP_SECONDS,
        max_samples=MAX_SAMPLES, confidence=CONFIDENCE,
        group_seconds=REVIEW_WINDOW_SECONDS, max_review_calls=MAX_GEMINI_CALLS,
        native_crops=NATIVE_CROPS,
    )
    print("Evidence folder:", RUN_DIR)
    PID = workflow.start(
        RUN_DIR, python_dmd=PYTHON_DMD, python_world=PYTHON_WORLD,
        python_gemini=PYTHON_GEMINI, use_gemini=RUN_GEMINI,
    )
    workflow.wait(RUN_DIR, timeout=1200)

## 4 · Inspect the summary

Counts describe sampled frames and proposal regions, not unique animals, calibrated probabilities or accuracy. Incomplete decoding is flagged. A short preview is not full-video coverage.

In [ ]:
if RUN_DIR is None:
    print("Upload and run one video above to see the summary.")
else:
    summary = workflow.read(RUN_DIR / "summary.json")
    cards = workflow.build_cards(RUN_DIR)
    display(Markdown(
        f"**Samples:** {summary['samples']} · **Review windows:** {summary['review_windows']} · "
        f"**Human-confirmed frames:** {summary['human_confirmed_frames']} · "
        f"**Context regions displayed:** {summary['context_regions_shown']}\n\n"
        f"**Whole source covered:** {summary['covers_whole_source']} · "
        f"**Requested window decoded completely:** {summary['complete_requested_window']}"
    ))
    print("Gemini statuses:", summary["gemini_statuses"])
    for card in cards:
        print(card["card_id"], "source", round(card["source_seconds"], 3), "s",
              "| animal proposals", card["animal_proposals"],
              "| Gemini", card["gemini_advisory"],
              "| human", card["human_decision"],
              "| context shown", card["shown_context_regions"])
    print("JSON summary:", RUN_DIR / "summary.json")
    print("Raw evidence:", RUN_DIR / "manifest.json")
    print("Review decisions:", RUN_DIR / "review-cards.json")

## 5 · Review an exact frame

Select a card and inspect the source image. **These buttons record your actual decision**; Run All does not press them for you. Confirmation applies to the selected source frame only—even if another timestamp happens to have identical pixels. Other samples stay unconfirmed.

A positive animal decision reveals only the separately supported person/vehicle regions. A no/uncertain decision keeps them hidden. Use Clear decision to undo your selection. Unreviewed context stays hidden even if the animal is confirmed.

In [ ]:
if RUN_DIR is None:
    print("The upload panel will show the review buttons when the video is ready.")
else:
    workflow.review_widget(RUN_DIR)

## 6 · View or refresh the annotated video

Run this cell again after clicking a review button. The exported H.264 MP4 contains sampled source frames, animal proposals, current review state and only the permitted context overlays. It is an offline preview, not live tracking. The JSON evidence preserves every original sampled-frame hash and timestamp.

In [ ]:
if RUN_DIR is None:
    print("The upload panel will show the output video; use Refresh annotated video after review.")
else:
    output_video, summary = workflow.export_video(RUN_DIR)
    display(Video(str(output_video), embed=True, width=960))
    print("MP4:", output_video)
    print("Summary:", RUN_DIR / "summary.json")
    print("Human decisions:", RUN_DIR / "human-decisions.json")
    print("Raw detector and review evidence remain in:", RUN_DIR)

## Interpretation and limits

- Green boxes are DMD animal proposals; they can be wrong. A model negative does not prove absence.
- Cyan person / amber vehicle regions require exact-frame human animal confirmation and a supported context check.
- One representative frame is sent for advice per window, with at most two person and two vehicle targets. Other proposals remain archived and unverified.
- This pilot requires a GPU for its detector workers. It makes bounded Gemini requests, stores failures, and keeps source provenance.
- It does not assess rescue urgency, identify people, estimate proximity, create incidents, dispatch responders or control a drone.
- Original footage stays local to the notebook server; selected review images are sent to the configured Gemini service.

---

# Historical comparison report

The following cells display the earlier fixed development experiments. They do not describe the new VIDEO run above.

> Historical research snapshot. These saved results do not describe a newly uploaded video.

# Tested recommendation: keep DMD; make Gemini advisory

**The decisive result is tiny-animal retention. A Gemini negative must not delete a DMD candidate.** The practical default is offline, operator-reviewed evidence windows, with selective AI review and class-specific context checks.

Tested **205 video frames across7 sources**, grouped into **19 windows**, four new review modes, a mixed-scene probe and three tiny-seal presentation checks. **104 new requests; 3 unusable outputs.** Existing four-model detection scores and five animal-evidence policies were also compared.

## New workflow comparison

| Method | Positive anchors | Control flags | Median call |
|---|---:|---:|---:|
| Single whole | 13/13 | 0/5 | 4.04s |
| Single + native crops | 11/13 | 0/5 | 4.41s |
| Three whole frames | 12/13 | 0/5 | 6.41s |
| Three + native crops | 10/13 | 1/5 | 7.87s |

Three native-mode outputs were unusable under the response/evidence checks; they remain failures, not silent retries. The single-whole method used55,540 reported total tokens, single-native44,473, burst-whole50,530 and burst-native100,370. Token totals are not dollar-cost estimates. These are manually screened window/anchor diagnostics, **not recall, specificity or box accuracy**. All methods review the same fixed midpoint or first/midpoint/last samples. Native adds candidate crops. Single and burst outputs refer only to their reviewed frames.

## Tiny annotated aerial seals: the stress check that changes the decision

| Input presentation | Mammal-positive images | Uncertain | No | Failed |
|---|---:|---:|---:|---:|
| single_whole | 0/8 | 0 | 8 | 0 |
| single_native | 0/8 | 3 | 5 | 0 |
| tight_native | 0/8 | 5 | 3 | 0 |

These are8 positive NOAA images with14 reference boxes. Native uses detector proposals, not ground-truth crops. The tighter presentation changes padding, crop budget and resize together; upsampling adds no detail. Presence is scored, not localization. There is no matched-domain negative set.

## Detector choice

| Model, tiled + whole | Matched seal boxes /14 | Unmatched proposals |
|---|---:|---:|
| dmd | 10/14 | 23 |
| grounding_dino | 2/14 | 37 |
| yolo_world | 0/14 | 18 |
| yoloe26 | 6/14 | 86 |

DMD whole-frame-only matched 0/14 in that cached subset. Preserve tiling for small animals; faster whole-frame processing is not an equivalent replacement.

YOLO-World remains a context proposer rather than the animal detector. The prior class-specific context test retained19 person-display frames and15 vehicle-display frames in the ground regression, while rejecting23 inspected surf/rock proposals. Those are display diagnostics, not accuracy.

## Annotated person/vehicle comparison (cached)

| Model | Person matches /30 | Vehicle matches /6 | Extra person / vehicle proposals |
|---|---:|---:|---:|
| dmd | 11 | 0 | 1 / 0 |
| grounding_dino | 25 | 6 | 9 / 1 |
| yolo_world | 23 | 5 | 2 / 0 |
| yoloe26 | 19 | 4 | 3 / 1 |

Grounding DINO matched more context references, while YOLO-World produced fewer extra context proposals and was faster in the prior T4 benchmark. This small inspected subset does not establish generalization.

## Cached animal-gate policy tradeoffs

| Rule | Surf active frames /31 | Beach active frames /38 | Ground active frames /24 |
|---|---:|---:|---:|
| crop_only | 20 | 6 | 24 |
| scene_only | 20 | 6 | 24 |
| same_check_agreement | 20 | 6 | 24 |
| two_scene_checks | 7 | 0 | 18 |
| agreement_or_repeated | 20 | 6 | 24 |

These are state replays using identical prior model responses, not new inference or accuracy. More stringent rules can suppress ambiguous positives but also delay/drop useful animal evidence.

## Practical configuration

1. Retain tiled DMD animal candidates and the original video. Group evidence into short review windows; do not count windows as animals.
2. Use a whole-view advisory check on a representative frame. Add native crops for uncertainty or disagreement. A negative AI judgment does not erase detector evidence.
3. Present an unconfirmed review card. A reviewer confirms animal presence for a specific immutable frame.
4. Show person/vehicle boxes only for that confirmed frame, with class-specific context verification. Do not carry approval to another scene or unreviewed frame.
5. Keep live streaming, automatic alerts and domain fine-tuning as separate validation work.

Adaptive routing replay would require 25 calls for these19 windows and preserves 6 detector candidate windows that the used AI responses do not confirm. This is an offline replay, not a deployed adaptive timing test. On the shared165-frame subset, the prior per-frame configuration requires117 model requests versus19 for this adaptive replay; outputs and review granularity differ, so this is a request-count comparison, not a measured speedup.

## Timing, boundaries and remaining work

- Fresh6-frame T4 timing, full: median 0.234s; 4K subset median 0.235s. Two synchronized repeats after warm-up; source decode/loading excluded.
- Fresh6-frame T4 timing, full_plus_tiles: median 1.986s; 4K subset median 2.002s. Two synchronized repeats after warm-up; source decode/loading excluded.
- Whole and native burst modes were tested across an injected visual cut. Context is joined to animal evidence only within the same reviewed frame. This is not natural scene-cut validation.
- Cached policy replays show the latency/retention tradeoff of stricter agreement and repeated checks; all underlying responses remain the same.
- No independently annotated aerial marine-animal-plus-human/vehicle positive was found. Dog-human footage is an aerial proxy; the seal release is ground-level.
- No training or independent flight-level fine-tuning evaluation was performed. No live2FPS, identity tracking, app integration or deployment claim.

## Reproduce

practical_options_prepare_v1.py freezes inputs and selection; practical_options_review_v1.py runs the four modes; practical_options_timing_v1.py measures detector cost; cut/seal/tight probe scripts run the stress checks; practical_options_archive_v1.py audits and scores. Exact request images and media references are under ~/aerial-data/practical-options-v1. Reports, plans, decisions and cards are under research/aerial_detector/results/practical-options-v1. Notebook cells display cached results only.

![Research evidence](../assets/tiny-seal-evaluation.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

![Research evidence](../assets/dog-review-card.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

![Research evidence](../assets/beach-review-card.jpg)

Attribution: [CREDITS.json](../assets/CREDITS.json).

In [ ]:
from pathlib import Path
import json
root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'research/aerial_detector/results').is_dir())
research=root/'research/aerial_detector'
print(json.dumps(json.loads((research/'results/practical-options-v1/report.json').read_text()),indent=2))